# Rental bikes count prediction: Dataset EDA

### Goal
We predict the **hourly total number of rented bikes (`cnt`)** in the Capital
Bikeshare system (Washington, D.C., 2011-2012) from calendar and weather
features of the same hour. `cnt` is the sum of casual and registered users.

**Why it matters.** Operators need demand estimates to decide how many bikes
to place at stations and when to rebalance them.

**Research question.** How much does random k-fold cross-validation
overestimate the accuracy of hourly demand predictions compared with
chronological validation, and does this gap depend on the type of model?

### What this notebook does
Inspects the size and types of the data, missing values, the distribution of
the target, the time structure (gaps, ordering), relationships between the
features and the target, and possible leakage. It ends with a table of
decisions on what to keep, drop or encode, and why.

### Data
Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning
Repository. https://doi.org/10.24432/C5W894 (CC BY 4.0).

## 1 - Setup

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
PATH = "data/hour.csv"

In [5]:
df = pd.read_csv(PATH)
print(df.shape)

(17379, 17)


## 2 - ...

In [6]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  str    
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), str(1)
memory usage: 2.3 MB


In [8]:
print(df.isna().sum())
print("duplicate rows:", df.duplicated().sum())

instant       0
dteday        0
season        0
yr            0
mnth          0
hr            0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64
duplicate rows: 0


In [9]:
df["dteday"] = pd.to_datetime(df["dteday"])
ts = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")

print("instant increasing:", df["instant"].is_monotonic_increasing)
print("timestamps increasing:", ts.is_monotonic_increasing)
print("timestamps unique:", ts.is_unique)

instant increasing: True
timestamps increasing: True
timestamps unique: True


In [10]:
print((df["casual"] + df["registered"] == df["cnt"]).all())

True


In [1]:
#checking